# Zero to Snowflake - Snowflake 入門

**学習内容**
1. 仮想ウェアハウスと設定
2. クエリ結果キャッシュの活用
3. 基本的なデータ変換テクニック
4. UNDROP によるデータ復元
5. リソースモニター
6. 予算管理（Budgets）
7. ユニバーサルサーチ

## セットアップ

セッションのクエリタグとコンテキスト（データベース・ロール）を設定します。

In [ ]:
ALTER SESSION SET query_tag = '{"origin":"sf_sit-is","name":"tb_zts","version":{"major":1, "minor":1},"attributes":{"is_quickstart":1, "source":"tastybytes", "vignette": "getting_started_with_snowflake"}}';

USE DATABASE tb_101;
USE ROLE accountadmin;

## 1. 仮想ウェアハウスと設定

> **ユーザーガイド**: https://docs.snowflake.com/ja/user-guide/warehouses-overview

仮想ウェアハウスは Snowflake データに対して分析を実行するための動的でスケーラブル、かつコスト効率に優れたコンピューティングリソースです。
基盤となる技術的な詳細を意識することなく、すべてのデータ処理ニーズを処理することを目的としています。

**主なパラメータ:**

| パラメータ | 説明 | デフォルト |
|---|---|---|
| `WAREHOUSE_SIZE` | クラスターあたりのコンピュートリソース量（X-Small〜6X-Large） | XSmall |
| `WAREHOUSE_TYPE` | `STANDARD`（汎用）または `SNOWPARK_OPTIMIZED`（メモリ集約型） | STANDARD |
| `AUTO_SUSPEND` | 自動停止までの非アクティブ時間（秒） | 600秒 |
| `INITIALLY_SUSPENDED` | 作成直後にサスペンド状態で開始するか | TRUE |
| `AUTO_RESUME` | クエリ送信時に自動再開するか | TRUE |

それでは、最初のウェアハウスを作成してみましょう！

In [ ]:
-- アクセス権限を持つ既存ウェアハウスを確認する
-- ウェアハウスの名前・状態（実行中/サスペンド）・タイプ・サイズなどが表示されます
SHOW WAREHOUSES;

SHOW WAREHOUSES の結果には、ウェアハウスの名前・状態（実行中またはサスペンド）・タイプ・サイズなどの属性が一覧表示されます。

Snowsight でもすべてのウェアハウスを表示・管理できます。
ウェアハウスページにアクセスするには:
- ナビゲーションメニューの **管理** における **コンピュート** へマウスオーバーし、 **ウェアハウス** リンクをクリックする

ウェアハウスページには、アカウント上のウェアハウスとその属性の一覧が表示されます。

In [ ]:
-- シンプルな SQL コマンドでウェアハウスを簡単に作成できます
CREATE OR REPLACE WAREHOUSE my_wh
    COMMENT = 'TastyBytes 用マイウェアハウス'
    WAREHOUSE_TYPE = 'standard'
    WAREHOUSE_SIZE = 'xsmall'
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 2
    SCALING_POLICY = 'standard'
    AUTO_SUSPEND = 60
    INITIALLY_SUSPENDED = true
    AUTO_RESUME = false;

ウェアハウスを作成したら、このワークシートがそのウェアハウスを使用するように指定する必要があります。
SQL コマンドまたは UI のどちらでも設定できます。

In [ ]:
-- ウェアハウスをセッションのアクティブウェアハウスとして設定する
USE WAREHOUSE my_wh;

### ウェアハウスの使用と再開

以下のクエリを実行すると**エラーになります**。
ウェアハウスがサスペンド状態で `AUTO_RESUME` が無効なためです。

クエリの実行やすべての DML 操作にはアクティブなウェアハウスが必要なため、
データからインサイトを得るにはウェアハウスを再開する必要があります。

エラーメッセージには SQL コマンドの実行を提案するヒントも含まれています: `ALTER WAREHOUSE MY_WH RESUME`。早速実行しましょう！

In [ ]:
-- ⚠️ このクエリはエラーになります（ウェアハウスがサスペンド中）
SELECT * FROM raw_pos.truck_details;

In [ ]:
-- ウェアハウスを再開する
ALTER WAREHOUSE my_wh RESUME;

-- 再度サスペンドした場合に手動再開が不要となるよう AUTO_RESUME を有効化する
ALTER WAREHOUSE my_wh SET AUTO_RESUME = TRUE;

In [ ]:
-- ウェアハウス再開後は正常にクエリが実行できます
SELECT * FROM raw_pos.truck_details;

### ウェアハウスのスケーリング

次に、Snowflake のウェアハウスのスケーラビリティの力を体験してみましょう。

Snowflake のウェアハウスはスケーラビリティと弾力性を考慮して設計されており、
ワークロードのニーズに応じてコンピュートリソースを柔軟に増減できます。

シンプルな `ALTER WAREHOUSE` 文でウェアハウスをオンザフライでスケールアップできます。

> **補足 - Adaptive Warehouse（Public Preview）**  
> ウェアハウスサイズの選択が不要な **Adaptive Warehouse** が現在 Public Preview として提供されています。
> ワークロードに応じてコンピュートリソースを自動的に最適化するため、サイズ選択の手間がなくなります。
>  
> - 公式ドキュメント: https://docs.snowflake.com/en/user-guide/warehouses-adaptive
> - 参考記事: https://dev.classmethod.jp/articles/snowflake-try-adaptive-warehouse/

In [ ]:
-- より集中的なワークロードに備えて XLarge にスケールアップ
ALTER WAREHOUSE my_wh SET warehouse_size = 'XLarge';

In [ ]:
-- トラックブランド別の売上を集計する
SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;

> **結果パネルの確認**: クエリ実行後、右上のツールバーで以下の操作ができます。
> - **検索**: 検索語句で結果をフィルタリングする
> - **カラム選択**: 結果に表示するカラムを有効/無効にする
> - **クエリ詳細**: SQL テキスト・返された行数・クエリID・実行ロール・ウェアハウスなど、クエリに関連する情報を表示する
> - **クエリ実行時間**: コンパイル・プロビジョニング・実行時間の内訳を表示する
> - **カラム統計**: 結果パネルのカラム分布に関連するデータを表示する (Notebookでは確認できず、SQLファイルで確認可能な項目です)
> - **結果のダウンロード**: 結果を CSV としてエクスポート・ダウンロードする

## 2. クエリ結果キャッシュの活用

> **ユーザーガイド**: https://docs.snowflake.com/ja/user-guide/querying-persisted-results

先ほどの「トラック別売上」クエリは、XLarge ウェアハウスでも実行に数秒かかりました。
同じクエリを再実行し、クエリ実行時間ペインで合計実行時間を確認してください。
初回実行では数秒かかったものが、次回実行ではほんの数百ミリ秒になっていることに気づくでしょう。
これがクエリ結果キャッシュの効果です。

クエリ履歴パネルを開き、初回と2回目の実行時間を比較してみてください。

**クエリ結果キャッシュの概要:**
- クエリ結果は **24 時間** 保持されます（クエリが実行されるたびにタイマーがリセット）
- キャッシュヒット時はコンピュートリソースをほぼ消費しないため、頻繁に実行されるレポートやダッシュボードのコスト削減に最適
- キャッシュはクラウドサービスレイヤーに存在し、**同一アカウント内のすべての仮想ウェアハウスとユーザーからグローバルにアクセス可能**

In [ ]:
-- 同じクエリを再実行 → キャッシュから即座に結果が返ります
-- クエリ履歴で初回と2回目の実行時間を比較してみてください
SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;

In [ ]:
-- 小さいデータセットを扱うのでクレジット節約のため XSmall にスケールダウン
ALTER WAREHOUSE my_wh SET warehouse_size = 'XSmall';

## 3. 基本的なデータ変換テクニック

ウェアハウスが設定され稼働中になったので、トラックメーカーの分布を把握したいと思います。
しかし、この情報は `VARIANT` データ型で年式、メーカー、モデルの情報が格納されている
`truck_build` カラムに埋め込まれています。

`VARIANT` データ型は半構造化データの一例です。OBJECT、ARRAY、他のVARIANT値などあらゆるデータ型を格納できます。
今回の場合、`truck_build` には `year`・`make`・`model` の3つの VARCHAR 値を持つ単一の OBJECT が格納されています。

これら3つのプロパティをそれぞれ独立したカラムに分離することで、シンプルで使いやすい分析が可能になります。

In [ ]:
-- truck_build VARIANT カラムの中身を確認する
SELECT truck_build FROM raw_pos.truck_details;

### ゼロコピークローニング（Zero Copy Cloning）

`truck_build` カラムのデータは常に同じフォーマットに従っています。
`make` のデータ品質分析をより簡単に行うために、別途カラムが必要です。
計画としては、truck テーブルの開発コピーを作成し、`year`・`make`・`model` の新しいカラムを追加してから、
`truck_build` VARIANT オブジェクトから各プロパティを抽出してこれらの新しいカラムに格納します。

Snowflake の強力なゼロコピークローニングにより、データベースオブジェクトの同一で完全に機能する
独立したコピーを、追加のストレージスペースを一切使用せずに即座に作成できます。

ゼロコピークローニングは Snowflake 独自のマイクロパーティションアーキテクチャを活用して、
クローンオブジェクトと元のコピーの間でデータを共有します。
どちらかのテーブルに変更を加えると、変更されたデータのみの新しいマイクロパーティションが作成されます。
これらの新しいマイクロパーティションは、クローンまたは元のオブジェクトのいずれか変更した側が所有します。
基本的に、一方のテーブルに加えられた変更は、もう一方に影響しません。

In [ ]:
-- truck テーブルのゼロコピークローンとして truck_dev を作成する
CREATE OR REPLACE TABLE raw_pos.truck_dev CLONE raw_pos.truck_details;

In [ ]:
-- クローンが正常に作成されたことを確認する
SELECT TOP 15 *
FROM raw_pos.truck_dev
ORDER BY truck_id;

### 新しいカラムの追加とデータの変換

安全な開発テーブルができたので `year`・`make`・`model` のカラムを追加し、  
`truck_build` の VARIANT カラムからデータを抽出して格納します。

In [ ]:
-- 新しいカラムを追加する
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS year  NUMBER;
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS make  VARCHAR(255);
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS model VARCHAR(255);

次に、`truck_build` カラムから抽出したデータで新しいカラムを更新します。

In [ ]:
-- コロン（:）演算子で truck_build から各プロパティを抽出し、新カラムに格納する
UPDATE raw_pos.truck_dev
SET
    year  = truck_build:year::NUMBER,
    make  = truck_build:make::VARCHAR,
    model = truck_build:model::VARCHAR;

In [ ]:
-- 3 つのカラムが正常に追加され、truck_build から抽出されたデータが格納されたことを確認する
SELECT year, make, model FROM raw_pos.truck_dev;

異なるメーカーの数を集計して、TastyBytes フードトラック車両の分布を把握しましょう
データ品質の問題が見つかるはずです。

In [ ]:
-- メーカー別の台数を集計する
-- ⚠️ 'Ford' と 'Ford_' が別々のメーカーとして扱われている！
SELECT
    make,
    COUNT(*) AS count
FROM raw_pos.truck_dev
GROUP BY make
ORDER BY make ASC;

In [ ]:
-- シンプルな UPDATE で 'Ford_' を 'Ford' に修正する
UPDATE raw_pos.truck_dev
    SET make = 'Ford'
    WHERE make = 'Ford_';

In [ ]:
-- make カラムが正常に更新されたことを確認する
SELECT truck_id, make
FROM raw_pos.truck_dev
ORDER BY truck_id;

### SWAP を使った本番環境へのプロモート

開発テーブルがクリーンになりました。`SWAP WITH` コマンドで 2 つのテーブルをアトミックに入れ替え、  
`truck_dev` を即座に新しい本番テーブルとして昇格させます。

In [ ]:
-- truck_details と truck_dev をアトミックに入れ替える
ALTER TABLE raw_pos.truck_details SWAP WITH raw_pos.truck_dev;

In [ ]:
-- SWAP 後の正確なメーカー分布を確認する（Ford_ が統合されているはず）
SELECT
    make,
    COUNT(*) AS count
FROM raw_pos.truck_details
GROUP BY make
ORDER BY count DESC;

### クリーンアップ

本番テーブルから不要な `truck_build` カラムを削除します。  
その後、開発用の `truck_dev` テーブルを削除するところを、**意図的に本番テーブルを誤ってドロップします**（次のセクションの練習です）。

In [ ]:
-- 古い truck_build カラムを削除する
ALTER TABLE raw_pos.truck_details DROP COLUMN truck_build;

-- ⚠️ 意図的に本番テーブルを誤ってドロップします！
DROP TABLE raw_pos.truck_details;

## 4. UNDROP によるデータ復元

大変です！誤って本番テーブル `truck_details` をドロップしてしまいました。😱

幸い、Snowflake の **タイムトラベル** 機能の `UNDROP` コマンドでドロップ前の状態に即座に復元できます。  
データ保持期間（デフォルト 24 時間）内であれば復元可能です。

In [ ]:
-- テーブルが存在しないことを確認する（エラーが表示されます）
DESCRIBE TABLE raw_pos.truck_details;

In [ ]:
-- UNDROP でテーブルをドロップ前の状態に復元する
UNDROP TABLE raw_pos.truck_details;

In [ ]:
-- テーブルが正常に復元されたことを確認する
SELECT * FROM raw_pos.truck_details;

In [ ]:
-- 今度は本物の開発テーブル truck_dev を安全にドロップする
DROP TABLE raw_pos.truck_dev;

## 5. リソースモニター

> **ユーザーガイド**: https://docs.snowflake.com/ja/user-guide/resource-monitors

コンピュートの使用量と支出の監視は、クラウドベースのワークフローにとって重要です。
Snowflake はリソースモニターを使用してウェアハウスのクレジット使用量を
シンプルかつわかりやすく追跡する方法を提供しています。

リソースモニターでは、クレジットクォータを定義し、定義した使用量のしきい値に達した際に
関連するウェアハウスに対して特定のアクションをトリガーできます。

**リソースモニターが実行できるアクション:**
- `NOTIFY`: 指定ユーザーまたはロールにメール通知を送信
- `SUSPEND`: しきい値に達した際に関連ウェアハウスをサスペンドします。(注: 実行中のクエリは完了が許可されます。)
- `SUSPEND_IMMEDIATE`: しきい値に達した際に関連ウェアハウスをサスペンドし、実行中のすべてのクエリをキャンセルします。

それでは、ウェアハウス my_wh 用のリソースモニターを作成しましょう。

まず Snowsight でアカウントレベルのロールを `ACCOUNTADMIN` に設定しましょう。
手順:
1. 画面左下のユーザーアイコンをクリックする
2. **ロールの切り替え** にカーソルを合わせる
3. ロールリストパネルから **ACCOUNTADMIN** を選択する

次に、ワークシートで `accountadmin` ロールを使用します。

In [ ]:
USE ROLE accountadmin;

-- my_wh 用のリソースモニターを作成する
-- 月間 100 クレジット、75% で通知・90% でサスペンド・100% で即時サスペンド
CREATE OR REPLACE RESOURCE MONITOR my_resource_monitor
    WITH CREDIT_QUOTA = 100
    FREQUENCY = MONTHLY   -- DAILY / WEEKLY / YEARLY / NEVER も指定可能
    START_TIMESTAMP = IMMEDIATELY
    TRIGGERS ON 75  PERCENT DO NOTIFY
             ON 90  PERCENT DO SUSPEND
             ON 100 PERCENT DO SUSPEND_IMMEDIATE;

In [ ]:
-- リソースモニターを my_wh に適用する
ALTER WAREHOUSE my_wh
    SET RESOURCE_MONITOR = my_resource_monitor;

## 6. 予算管理（Budgets）

> **ユーザーガイド**: https://docs.snowflake.com/ja/user-guide/budgets

前のステップでは、ウェアハウスのクレジット使用量を監視するリソースモニターを設定しました。
このステップでは、Snowflake のコスト管理をより包括的かつ柔軟に行うための予算管理（Budget）を作成します。

リソースモニターがウェアハウスとコンピュートの使用量に特化しているのに対して、
予算管理はあらゆる Snowflake オブジェクトやサービスのコストを追跡・制限し、
ドル金額が指定されたしきい値に達した際にユーザーへ通知するために使用できます。

予算 (バジェット) を作成する前に、アカウントのメールアドレスを確認しておく必要があります。

**メールアドレスの確認手順:**
1. 画面左下のユーザーアイコンをクリックする
2. **設定** をクリックする
3. **プロファイル** をクリックし、メールフィールドにメールアドレスを入力する
4. **保存** をクリックする
5. 届いたメールを確認し、指示に従ってメールアドレスを確認する
   - 数分経ってもメールが届かない場合は **確認メールを再送信** をクリックしてください

In [ ]:
-- 予算 (バジェット) を作成する
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET my_budget()
    COMMENT = 'Tasty Bytes 用マイ予算';

### Snowsight でのバジェット設定

**管理者** » **コスト管理** » **予算** に移動します。  
（ウェアハウス選択を求められた場合は `tb_dev_wh` を選択）

予算ページには現在の期間の支出に関する指標が表示されます。
画面中央には予測支出とともに現在の支出のグラフが表示されます。
画面下部には先ほど作成した **MY_BUDGET** 予算が表示されます。クリックして予算ページを開きます。

右上の **予算の詳細** をクリックすると詳細パネルが表示されます。
現在は監視中のリソースがないので、**編集** ボタンをクリックして設定を行います。

1. 予算名はそのまま
2. **予算** を `100` に設定する
3. 先ほど設定した確認済みメールアドレスを入力し、**次へ** をクリック
4. **リソース** をクリックしてリソースを追加する
   - **データベース** → **TB_101** → **ANALYTICS** スキーマのチェックボックスをオンにする
   - **ウェアハウス** → **TB_DE_WH** のチェックボックスをオンにする
   - **次へ** をクリックする
5. **保存** をクリックする

## 7. ユニバーサルサーチ

> **ユーザーガイド**: https://docs.snowflake.com/ja/user-guide/ui-snowsight-universal-search

ユニバーサルサーチを使用すると、アカウント内の任意のオブジェクトを簡単に検索できるほか、
マーケットプレイスのデータ製品、関連する Snowflake ドキュメント、コミュニティナレッジベースの記事も探索できます。

**ステップ 1 - オブジェクトの検索:**
1. ナビゲーションメニューの **検索** をクリックする
2. ユニバーサルサーチの UI が表示されます。最初の検索語句を入力しましょう。
3. 検索バーに `truck` と入力し、結果を観察する
   - 上部のセクションには、データベース・テーブル・ビュー・ステージなど、アカウント上の関連オブジェクトのカテゴリが表示されます
   - データベースオブジェクトの下には、関連するマーケットプレイスのリストとドキュメントのセクションが表示されます

**ステップ 2 - 自然言語検索:**

自然言語で探しているものを説明する検索語句も使用できます。
どのトラックフランチャイズにリピーター顧客が最も多いかを調べたい場合、以下のように検索できます。

```
どのトラックフランチャイズが最も忠実な顧客基盤を持っていますか？
```

「テーブルとビュー」セクションの **すべてを表示 >** ボタンをクリックすると、クエリに関連するすべてのテーブルとビューを表示できます。

ユニバーサルサーチは異なるスキーマから複数のテーブルとビューを返します。
各オブジェクトに対して関連するカラムも一覧表示されていることに注目してください。
これらはすべて、リピーター顧客についてのデータドリブンな回答を得るための優れた出発点となります。

## リセット

このビネットで作成したオブジェクトをクリーンアップします。

In [ ]:
DROP RESOURCE MONITOR IF EXISTS my_resource_monitor;
DROP TABLE IF EXISTS raw_pos.truck_dev;

-- truck_details をリセットする
CREATE OR REPLACE TABLE raw_pos.truck_details
AS
SELECT * EXCLUDE (year, make, model)
FROM raw_pos.truck;

DROP WAREHOUSE IF EXISTS my_wh;
ALTER SESSION UNSET query_tag;